In [ ]:
import numpy as np
import pandas as pd

In [ ]:
#Selecting the required columns from csv, changing datatype and keeping 'NA' filter as false to keep null values as ""

leie = pd.read_csv("C:/Docs/Healthcare-Payment-Integrity/data/UPDATED.csv", 
                 usecols=['LASTNAME', 'FIRSTNAME', 'MIDNAME', 'BUSNAME', 'GENERAL','SPECIALTY','UPIN', 'NPI', 'CITY', 'STATE', 'ZIP', 'EXCLTYPE', 'EXCLDATE', 'REINDATE', 'WAIVERDATE', 'WVRSTATE'],
                 dtype=str, na_filter=False
                 )
print(leie)

In [ ]:
# Knowing the csv structure

leie.info()

In [ ]:
#Filtering to check specific state and city

leie_ma = leie[(leie['STATE']=='MA') & (leie['CITY']=='LANCASTER')]
print(leie_ma)

In [ ]:
#To confirm the count for each NPI

print(leie['NPI'].value_counts().head(10))

In [ ]:
#Checking per NPI values to identify duplicates

leie_npi = leie[leie['NPI'] == '1225072028']
print(leie_npi)

In [ ]:
#To confirm how many usuable NPIs are there

placeholder = leie['NPI'] == '0000000000'
print(placeholder.sum())

is_char = leie['NPI'].str.len() == 10
print(is_char.sum())

is_digits = leie['NPI'].str.isdigit()
print(is_digits.sum())

is_valid = is_char & is_digits
print(is_valid.sum())

condition = [placeholder, is_valid]
labels = ['placeholder', 'valid']
leie['npi_category'] = np.select(condition,labels,default='other')


In [ ]:
#To check if any lastname is there which has null

filtered_lastname = leie[leie['LASTNAME'].isin(['NA','NAN','N/A','NULL','NONE'])]
print(filtered_lastname)


In [ ]:
# To check between individual vs business

has_last = leie['LASTNAME'] !=""
print(has_last.sum())

has_bus = leie['BUSNAME'] !=""
print(has_bus.sum())

has_ind = has_last & ~has_bus
print(has_ind.sum())

has_bname = has_bus & ~has_last
print(has_bname.sum())

has_both = has_last & has_bus
print(has_both.sum())

has_neither = ~has_last & ~has_bus
print(has_neither.sum())

In [ ]:
# To check the states MA & FL

both_states = leie['STATE'].isin(['MA','FL'])
print(both_states.sum())

In [ ]:
# combining all 3 to find the correcrt values to match NPI

combined = both_states & has_ind & ~placeholder
print(combined.sum())

In [ ]:
#Convert to Parquet

leie_cols = ['LASTNAME', 'FIRSTNAME', 'MIDNAME', 'BUSNAME', 'GENERAL','SPECIALTY','UPIN', 'NPI', 'CITY', 'STATE', 'ZIP', 'EXCLTYPE', 'EXCLDATE', 'REINDATE', 'WAIVERDATE', 'WVRSTATE']

leie[leie_cols].to_parquet('C:/Docs/Healthcare-Payment-Integrity/data/Processed/leie_exclusions.parquet', index=False)

In [ ]:
#Check the Parquet file

df = pd.read_parquet('C:/Docs/Healthcare-Payment-Integrity/data/Processed/leie_exclusions.parquet')
print(df)